<a href="https://colab.research.google.com/github/CaelMarshall/Crime-Data-in-Leeds-April-2024-GEOG5990M-/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:


!pip install contextily
#Instalss spatial plotting tools
!pip install geoplot
#Installs features that allow the addition of a north arrow to maps made
!pip install git+https://github.com/pmdscully/geo_northarrow.git
# imports pandas which is used for data manipulation and gives it a shorter alias which is used in further code.
import pandas as pd
#imports numpy for numerical operations and gives it a shorter alias for use in further code
import numpy as np
#Imports geopandas which is used for spatial data and allows the plotting of maps, points and polygons based on
#what the data shows
import geopandas as gpd
# imports matplotlib for plotting graphs and gives it a shorter alias for use in further code. Plt is used to control
#factors such as titles, labels and display functions of graphs
import matplotlib.pyplot as plt
#Used for adding coordinate reference systems and projections
import pyproj
# imports systems used to add basemaps to spatial plots
import contextily as ctx
# imports seaborn which is used to make nicer statistics visualisations and gives it a shorter alias for use
# in further code.
import seaborn as sns
#importsfeatures that allow for spatial plottng
import geoplot as gplt
# imports features that allow for the manipulation of coordinate referance system
import geoplot.crs as gcrs
# imports a function to add a north arrow to the map
from geo_northarrow import add_north_arrow

from google.colab import files
uploaded = files.upload()


import os
os.listdir()

# # Data downloaded from https://geoportal.statistics.gov.uk/maps/761ecd09b4124843b95511a242e2b1a1
 shp =gpd.read_file('LSOA_2021_EW_BGC_V5.shp')
leeds_shp =shp.loc[shp['LSOA21NM'].str.contains('Leeds'),:]
leeds_shp.to_file('Leeds.geojson')

lsoa = gpd.read_file("LSOA_2021_EW_BGC_V5.shp")
Imd = pd.read_csv("IMD_BY_LSOA.csv")
crime = pd.read_csv("April_Crime_Data.csv")

Imd.head()
crime.head()
Imd.info()
Imd.columns
crime.info()

leeds_shp.explore()
leeds_shp.info()
leeds_shp.crs
crime.info()

crime_gdf = gpd.GeoDataFrame(
    crime, geometry=gpd.points_from_xy(crime.Longitude, crime.Latitude))
crime_gdf = crime_gdf.set_crs(epsg = 4326)

crime_gdf = crime_gdf.to_crs(leeds_shp.crs)

crime_lsoa = gpd.sjoin(crime_gdf, leeds_shp, how="inner", predicate="within")
crime_lsoa.head()

crime_counts = crime_lsoa.groupby("LSOA21CD").size().reset_index(name="crime_count")
leeds_shp = leeds_shp.merge(crime_counts, on="LSOA21CD", how= "left")
leeds_shp["crime_count"] = leeds_shp["crime_count"].fillna(0)

 leeds_shp = leeds_shp.merge(Imd, left_on="LSOA21CD", right_on="LSOA code (2021)", how="left")
